# Fine-tuning Qwen3-0.6B for Voice Banking

This notebook fine-tunes the **Qwen3-0.6B** model using **Unsloth + LoRA** to act
as a low-latency tool-calling model for the BankCo voice assistant.

## What this notebook does

1. Loads the base Qwen3-0.6B model with Unsloth's fast patching
2. Applies LoRA adapters (r=16, ~1.67% of parameters trainable)
3. Formats multi-turn banking dialogues into the Qwen chat template
4. Trains for 1 epoch with SFTTrainer
5. Exports both LoRA adapters and merged 16-bit weights
6. Provides commands for GGUF conversion (llama.cpp deployment)

## Prerequisites

- NVIDIA GPU with ≥8 GB VRAM
- Install training dependencies: `pip install -r requirements-training.txt`
- Download the base model: [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B)
- Prepare your training data in JSONL format (see Data Format section below)

## Step 1 — Imports

In [ ]:
from unsloth import FastLanguageModel
import json
from datasets import Dataset, load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

## Step 2 — Load Model & Apply LoRA

Update `MODEL_PATH` to point to your local copy of Qwen3-0.6B.

In [ ]:
# === CONFIGURE THESE PATHS ===
MODEL_PATH = "./model/Qwen3-0.6B"                       # Path to base model
TRAIN_DATA = "./data/train_data.jsonl"             # Path to training JSONL
OUTPUT_DIR = "./outputs/training_checkpoints"      # Checkpoint directory
LORA_SAVE_DIR = "./outputs/qwen3_06b_lora"        # LoRA adapter output
MERGED_SAVE_DIR = "./outputs/qwen3_06b_merged"    # Merged 16-bit output

max_seq_length = 3072

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=max_seq_length,
    load_in_4bit=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

## Step 3 — Prepare Training Data

### Expected JSONL Format

Each line should be a JSON object with a `messages` array following OpenAI's
multi-turn format with `tool_calls`:

```json
{"messages": [
  {"role": "user", "content": "I want to check my savings balance"},
  {"role": "assistant", "tool_calls": [{"function": {"name": "check_balance", "arguments": {"account_type": "savings"}}}]}
]}
```

The formatting function below converts these into the compact JSON output
format that the model learns to generate (no explicit tool schema injection).

In [ ]:
def format_multi_turn_implicit(sample):
    """Convert a multi-turn dialogue sample into the Qwen chat template format.
    
    Assistant turns with tool_calls are converted to compact JSON strings:
        {"name": "function_name", "arguments": {"key": "value"}}
    """
    try:
        raw_turns = sample["messages"]
        
        static_system = {
            "role": "system",
            "content": (
                "You are a low-latency banking voice assistant. Analyze the conversation history "
                "and output ONLY a valid JSON object matching the required tool call structure."
            )
        }
        
        cleaned_turns = [static_system]
        
        for turn in raw_turns:
            if turn["role"] == "user":
                cleaned_turns.append({"role": "user", "content": turn["content"]})
            elif turn["role"] == "assistant":
                if "tool_calls" in turn and turn["tool_calls"]:
                    func_block = turn["tool_calls"][0]["function"]
                    t_name = func_block["name"]
                    t_args = {k: v for k, v in func_block.get("arguments", {}).items() if v is not None}
                else:
                    t_name = "intent_unclear"
                    t_args = {}
                
                json_text_output = json.dumps({"name": t_name, "arguments": t_args})
                cleaned_turns.append({"role": "assistant", "content": json_text_output})
        
        text = tokenizer.apply_chat_template(cleaned_turns, tokenize=False, add_generation_prompt=False)
        return {"text": text}
    
    except Exception:
        return None


print("Loading training data...")
raw_dataset = load_dataset("json", data_files={"train": TRAIN_DATA}, split="train")
splits = raw_dataset.train_test_split(test_size=0.1, seed=345)

def process_split(dataset_split):
    processed = []
    for row in dataset_split:
        fmt = format_multi_turn_implicit(row)
        if fmt:
            processed.append(fmt)
    return Dataset.from_list(processed)

print("Formatting sequences with native chat template...")
train_dataset = process_split(splits["train"])
val_dataset = process_split(splits["test"])

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

## Step 4 — Fine-tune with SFTTrainer

Training hyperparameters:
- **Batch size**: 2 × 4 gradient accumulation = effective batch 8
- **Learning rate**: 5e-5 with cosine schedule
- **Epochs**: 1 (sufficient for ~5k examples)
- **Trainable params**: ~10M / 606M (1.67%)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=20,
        num_train_epochs=1,
        learning_rate=5e-5,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=100,
        optim="adamw_8bit",
        weight_decay=0.05,
        lr_scheduler_type="cosine",
        seed=783,
        output_dir=OUTPUT_DIR,
        report_to="none",
        dataloader_num_workers=2,
        dataloader_pin_memory=True,
    ),
)

print("Starting fine-tuning...")
trainer.train()

## Step 5 — Export Model Artifacts

Saves both:
1. **LoRA adapters** — lightweight, can be loaded on top of the base model
2. **Merged 16-bit model** — ready for GGUF conversion with llama.cpp

In [ ]:
import shutil

# Clean previous exports to prevent weight collisions
print("Cleaning previous exports...")
shutil.rmtree(LORA_SAVE_DIR, ignore_errors=True)
shutil.rmtree(MERGED_SAVE_DIR, ignore_errors=True)

# Save LoRA adapters
print("Saving LoRA adapters...")
model.save_pretrained(LORA_SAVE_DIR)
tokenizer.save_pretrained(LORA_SAVE_DIR)

# Merge and save full 16-bit model
print("Merging and saving 16-bit weights...")
model.save_pretrained_merged(MERGED_SAVE_DIR, tokenizer, save_method="merged_16bit")

print("All exports completed successfully!")

## Step 6 — Convert to GGUF for llama.cpp

Run these commands in your terminal (not in the notebook).

### Convert merged model to GGUF

```bash
python llama.cpp/convert_hf_to_gguf.py ./outputs/qwen3_06b_merged \
    --outfile ./outputs/qwen3_06b_voice_banking_f16.gguf \
    --outtype f16 \
    --split-max-size 50G
```

### Start the inference server

```bash
llama-server \
  -m ./outputs/qwen3_06b_voice_banking_f16.gguf \
  --port 7002 \
  -ngl 99 \
  -c 2048 \
  --predict 64 \
  --temp 0.0 \
  -rea off \
  --no-context-shift \
  --min-p 0.05
```

Then run the voice assistant:

```bash
python app.py --model qwen3_06b_voice_banking_f16.gguf --port 7002
```

---

## Appendix — Training Data Augmentation

The cell below randomises numerical values (card numbers, amounts) in the
training data while maintaining consistency between user utterances and
assistant tool calls. This prevents the model from memorising specific
numbers from the training set.

In [ ]:
import json
import random

def process_structural_jsonl(input_filepath, output_filepath):
    """Randomise numerical entities (card numbers, amounts) in training data.
    
    Performs symmetrical replacement: if a number appears in the user text
    AND the tool call arguments, both are replaced with the same random value.
    Numbers written as words (e.g., 'fifty dollars') are preserved as-is.
    """
    processed_count = 0
    modified_count = 0
    skipped_word_count = 0
    
    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', encoding='utf-8') as outfile:
         
        for line_num, line in enumerate(infile, 1):
            line = line.strip()
            if not line:
                continue
            
            try:
                data = json.loads(line)
            except Exception:
                outfile.write(line + '\n')
                continue
            
            messages = data.get("messages", [])
            
            # Collect all user text for grounding validation
            user_text = ""
            for msg in messages:
                if msg.get("role") == "user":
                    user_text += str(msg.get("content", ""))
            
            replacements = {}
            
            # Extract and validate numerical entities
            for msg in messages:
                if msg.get("role") == "assistant" and "tool_calls" in msg and msg["tool_calls"]:
                    func_block = msg["tool_calls"][0].get("function", {})
                    args = func_block.get("arguments", {})
                    
                    if isinstance(args, dict):
                        for key, val in args.items():
                            if val is None or val == "":
                                continue
                            
                            if key in ["card_last_four"]:
                                old_val = str(val)
                                if old_val in user_text:
                                    if old_val not in replacements:
                                        replacements[old_val] = str(random.randint(1000, 9999))
                                else:
                                    skipped_word_count += 1
                                    
                            elif key in ["amount", "transaction_amount"]:
                                try:
                                    float_val = float(val)
                                    variants = [
                                        str(int(float_val)),
                                        f"{float_val:.1f}",
                                        f"{float_val:.2f}",
                                        str(val)
                                    ]
                                    
                                    if any(v in user_text for v in variants):
                                        rand_val = round(random.uniform(15.0, 1900.0), 2)
                                        new_int_str = str(int(rand_val))
                                        new_float_str = f"{rand_val:.2f}"
                                        
                                        replacements[variants[0]] = new_int_str
                                        replacements[variants[1]] = new_float_str
                                        replacements[variants[2]] = new_float_str
                                        replacements[variants[3]] = new_float_str
                                    else:
                                        skipped_word_count += 1
                                except ValueError:
                                    pass
            
            # Symmetrically replace entities in both user text and tool args
            if replacements:
                modified_count += 1
                for msg in messages:
                    if msg.get("role") == "user" and isinstance(msg.get("content"), str):
                        content = msg["content"]
                        for old_str, new_str in sorted(replacements.items(), key=lambda x: len(x[0]), reverse=True):
                            content = content.replace(old_str, new_str)
                        msg["content"] = content
                        
                    if msg.get("role") == "assistant" and "tool_calls" in msg and msg["tool_calls"]:
                        func_block = msg["tool_calls"][0].get("function", {})
                        args = func_block.get("arguments", {})
                        if isinstance(args, dict):
                            for k, v in args.items():
                                v_str = str(v)
                                if v_str in replacements:
                                    new_str_val = replacements[v_str]
                                    if isinstance(v, (float, int)):
                                        args[k] = float(new_str_val)
                                    else:
                                        args[k] = new_str_val
                            func_block["arguments"] = args
            
            outfile.write(json.dumps(data) + '\n')
            processed_count += 1
            
    print(f"Total lines processed: {processed_count}")
    print(f"Successfully randomized (Digit matches): {modified_count} rows.")
    print(f"Preserved original truth (Words/Missing): {skipped_word_count} entities.")


# === CONFIGURE THESE PATHS ===
INPUT_FILE = "./data/train_data_original.jsonl"
OUTPUT_FILE = "./data/train_data.jsonl"

process_structural_jsonl(INPUT_FILE, OUTPUT_FILE)